# Home Credit: understand the data first
This notebook completes our first task: inspecting the downloaded data and recording what needs attention before preprocessing.

Read the short explanation above each code cell, then look at its output. All numbers below come from your local files. The original CSVs are never modified.

**Plan:** load → understand columns → check missing values → check duplicates → inspect the target → investigate unusual values → record next steps.


## 1. Load the training table
`pandas` works with tables. `pd.read_csv()` loads a file into a DataFrame named `df`. `Path` locates the folder. One row represents one current loan application.

We use `application_train.csv` because it includes the recorded outcome (`TARGET`). The Kaggle test file does not include that outcome.


In [9]:
from pathlib import Path
import pandas as pd

# Locate the project from either ProJect/ or ProJect/notebooks/.
project_root = next(
    (candidate for candidate in [Path.cwd(), *Path.cwd().parents]
     if (candidate / "data" / "raw" / "home-credit-default-risk" / "application_train.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open this notebook from inside the ProJect folder.")
data_folder = project_root / "data" / "raw" / "home-credit-default-risk"

df = pd.read_csv(data_folder / "application_train.csv")
print(f"Loaded {df.shape[0]} applications and {df.shape[1]} columns.")
df[["SK_ID_CURR", "AMT_INCOME_TOTAL", "AMT_CREDIT", "TARGET"]].head()


Loaded 307511 applications and 122 columns.


,SK_ID_CURR,AMT_INCOME_TOTAL,AMT_CREDIT,TARGET
0,100002,202500.0,406597.5,1
1,100003,270000.0,1293502.5,0
2,100004,67500.0,135000.0,0
3,100006,135000.0,312682.5,0
4,100007,121500.0,513000.0,0


## 2. What are the other files for?
We list the files without loading every large table into memory. Later we can summarize historical records per application and join the summaries to the main table. We must verify that historical features were available at the time of the prediction.

- `previous_application`: earlier Home Credit applications.
- `installments_payments`: installment payment records.
- `bureau` and `bureau_balance`: other credit and monthly credit status records.
- `credit_card_balance` and `POS_CASH_balance`: monthly historical account snapshots.
- `HomeCredit_columns_description`: a reference dictionary, not model input.
- `sample_submission`: the format expected for Kaggle predictions, not training features.


In [2]:
files = pd.DataFrame([
    {"file": p.name, "size_MB": round(p.stat().st_size / 1_000_000, 2)}
    for p in sorted(data_folder.glob("*.csv"))
])
display(files)
test_columns = pd.read_csv(data_folder / "application_test.csv", nrows=0).columns
print("TARGET in training file:", "TARGET" in df.columns)
print("TARGET in Kaggle test file:", "TARGET" in test_columns)


,file,size_MB
0,application_test.csv,26.57
1,application_train.csv,166.13
2,bureau.csv,170.02
3,bureau_balance.csv,375.59
4,credit_card_balance.csv,424.58
5,HomeCredit_columns_description.csv,0.04
6,installments_payments.csv,723.12
7,POS_CASH_balance.csv,392.70
8,previous_application.csv,404.97
9,sample_submission.csv,0.54


TARGET in training file: True
TARGET in Kaggle test file: False


## 3. Read column meanings
The dictionary's `Row` column contains feature names. We create a True/False filter with `.isin()`, then use `.loc[rows, columns]` to keep matching definitions. This only reads the dictionary.

Example: if our selected names include `AMT_CREDIT`, its definition is kept. An unselected name is skipped. We also restrict the dictionary to the application table to avoid definitions from other tables.


In [ ]:
selected_columns = [
    "SK_ID_CURR",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "NAME_CONTRACT_TYPE",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "TARGET"
]



column_dictionary = pd.read_csv(
    data_folder / "HomeCredit_columns_description.csv",
    encoding="cp1252"
)

selected_definitions = column_dictionary.loc[
    (column_dictionary["Row"].isin(selected_columns))
    & (column_dictionary["Table"] == "application_{train|test}.csv"),
    ["Table", "Row", "Description"],
]





## TO MAKE THE DESCRIPTION APPEARS ALL OF IT WE USED THE BELOW CODE



with pd.option_context("display.max_colwidth", None):
    display(selected_definitions)


,Table,Row,Description
0,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample
1,application_{train|test}.csv,TARGET,"Target variable (1 - client with payment difficulties: he/she had late payment more than X days on at least one of the first Y installments of the loan in our sample, 0 - all other cases)"
2,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving
7,application_{train|test}.csv,AMT_INCOME_TOTAL,Income of the client
8,application_{train|test}.csv,AMT_CREDIT,Credit amount of the loan
9,application_{train|test}.csv,AMT_ANNUITY,Loan annuity
17,application_{train|test}.csv,DAYS_BIRTH,Client's age in days at the time of application
18,application_{train|test}.csv,DAYS_EMPLOYED,How many days before the application the person started current employment


## 4. Inspect column types
`info()` shows the column names, nonmissing counts, types, and memory use. `int64` means whole numbers; `float64` allows decimal values; `object` usually holds text here.

A number is not automatically a useful numeric input: `SK_ID_CURR` is an identifier used for joins, not an amount or measurement. We will exclude it from model features.


In [ ]:
df.info(verbose =True, show_counts=True)
df["AMT_ANNUITY"].dtype



<class 'pandas.DataFrame'>
RangeIndex: 307511 entries, 0 to 307510
Data columns (total 122 columns):
 #    Column                        Non-Null Count   Dtype  
---   ------                        --------------   -----  
 0    SK_ID_CURR                    307511 non-null  int64  
 1    TARGET                        307511 non-null  int64  
 2    NAME_CONTRACT_TYPE            307511 non-null  str    
 3    CODE_GENDER                   307511 non-null  str    
 4    FLAG_OWN_CAR                  307511 non-null  str    
 5    FLAG_OWN_REALTY               307511 non-null  str    
 6    CNT_CHILDREN                  307511 non-null  int64  
 7    AMT_INCOME_TOTAL              307511 non-null  float64
 8    AMT_CREDIT                    307511 non-null  float64
 9    AMT_ANNUITY                   307499 non-null  float64
 10   AMT_GOODS_PRICE               307233 non-null  float64
 11   NAME_TYPE_SUITE               306219 non-null  str    
 12   NAME_INCOME_TYPE              307511 no

## 5. Measure missing values
`isna()` marks missing cells as True. `sum()` counts them per column. `mean() * 100` gives the percentage. For example, 2 missing values among 10 records is 20%.

Missing is different from zero. Missing car age might reflect no car, for example; replacing it blindly with zero would mean a new car. We inspect definitions before deciding what to do.


In [ ]:
missing_values = df.isna().sum()
print("Missing values per column:")
display(missing_values.sort_values(ascending=False).head(15))

missing_values_percentage = (missing_values / len(df)) * 100
print("Missing percentage per column:")
display(missing_values_percentage.sort_values(ascending=False).head(15).round(2))


Missing values per column:


COMMONAREA_AVG              214865
COMMONAREA_MODE             214865
COMMONAREA_MEDI             214865
NONLIVINGAPARTMENTS_MEDI    213514
NONLIVINGAPARTMENTS_MODE    213514
NONLIVINGAPARTMENTS_AVG     213514
FONDKAPREMONT_MODE          210295
LIVINGAPARTMENTS_AVG        210199
LIVINGAPARTMENTS_MEDI       210199
LIVINGAPARTMENTS_MODE       210199
FLOORSMIN_MODE              208642
FLOORSMIN_AVG               208642
FLOORSMIN_MEDI              208642
YEARS_BUILD_AVG             204488
YEARS_BUILD_MODE            204488
dtype: int64

Missing percentage per column:


COMMONAREA_AVG              69.87
COMMONAREA_MODE             69.87
COMMONAREA_MEDI             69.87
NONLIVINGAPARTMENTS_MEDI    69.43
NONLIVINGAPARTMENTS_MODE    69.43
NONLIVINGAPARTMENTS_AVG     69.43
FONDKAPREMONT_MODE          68.39
LIVINGAPARTMENTS_AVG        68.35
LIVINGAPARTMENTS_MEDI       68.35
LIVINGAPARTMENTS_MODE       68.35
FLOORSMIN_MODE              67.85
FLOORSMIN_AVG               67.85
FLOORSMIN_MEDI              67.85
YEARS_BUILD_AVG             66.50
YEARS_BUILD_MODE            66.50
dtype: float64

## 6. Check duplicate records
`duplicated()` marks repeated entries after their first occurrence. Whole-row duplicates and repeated IDs are different: the same ID with different income values would repeat an ID without repeating an entire row. Investigate before deleting.


In [31]:
# Check whether entire records repeat.
duplicate_rows = df.duplicated().sum()
print("Duplicate rows:", duplicate_rows)

# Check whether application identifiers repeat, even if other values differ.
duplicate_ids = df["SK_ID_CURR"].duplicated().sum()
print("Repeated application IDs:", duplicate_ids)

# Check whether an application identifier is missing.
missing_ids = df["SK_ID_CURR"].isna().sum()
print("Missing application IDs:", missing_ids)


Duplicate rows: 0
Repeated application IDs: 0
Missing application IDs: 0


## 7. Inspect the outcome we want to predict
The dictionary defines TARGET=1 as payment difficulties according to its specified installment criteria; TARGET=0 covers all other cases. This is not a provided numeric credit score.

`value_counts()` counts each outcome; `normalize=True` gives proportions. If most outcomes are zero, a model can obtain high accuracy by predicting zero for everybody, while missing the difficult cases.


In [ ]:
target_counts = df["TARGET"].value_counts(dropna=False)
display(target_counts)
print("missing TARGET values:", df["TARGET"].isna().sum())


no_of_targetspercent = df["TARGET"].value_counts() / len(df)  *100
no_of_targetspercent


TARGET
0    282686
1     24825
Name: count, dtype: int64

missing TARGET values: 0


TARGET
0    91.927118
1     8.072882
Name: count, dtype: float64

## 8. Inspect numeric ranges and text categories
`describe()` shows nonmissing count, mean, standard deviation, minimum, quartiles, and maximum. `50%` is the median. `.T` turns the summary so each feature occupies a row.

Large values need investigation; they are not automatically errors. Text category counts help us discover rare entries and missing categories.


In [54]:
numeric_columns = df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_BIRTH", "DAYS_EMPLOYED"]].describe().T.round(2)
numeric_columns


df["NAME_INCOME_TYPE"].value_counts(dropna=False)
df["NAME_CONTRACT_TYPE"].value_counts(dropna=False)

NAME_CONTRACT_TYPE
Cash loans         278232
Revolving loans     29279
Name: count, dtype: int64

## 9. Investigate date offsets and unusually large values
`DAYS_BIRTH` and `DAYS_EMPLOYED` describe time relative to the application. A negative birth offset is expected: -10,950 days corresponds to roughly 30 years before application.

We inspect positive employment offsets and the repeated value 365243, which would imply about 1,000 years. Such a value should be treated as a placeholder candidate rather than literal employment duration. We will handle it explicitly during cleaning, preserving a flag if useful. We do not convert every positive value without inspection.


In [ ]:
unusual_employment = df["DAYS_EMPLOYED"] == 365243
unusual_employment.sum()


df.loc[unusual_employment, "NAME_INCOME_TYPE"].value_counts(dropna=False)

NAME_INCOME_TYPE
Pensioner     55352
Unemployed       22
Name: count, dtype: int64

: 

## 10. Save the reports and record what we learned
We save small inspection reports in `reports/data_understanding/`. These are summaries; the source data stays untouched. The next notebook will clean confirmed placeholders, split the labeled table, and fit preprocessing only on training rows.
